# Test an orthophoto-conditioned (v6) mesh-transformer checkpoint

The v6 sibling of `notebooks/test-mesh-normal.ipynb`. It loads the **same** splits
the run trained against, with the same config, seed and split scheme, and the same
"skip buildings without imagery" filter. Then it generates LOD2 for every building
**three times**, once per post-hoc `image_mode`:

| mode | what the model sees | what it answers |
|---|---|---|
| `true` | the building's own crop | the model as deployed |
| `shuffled` | the crop of the building half the split away | is the image used *as this building's image*? |
| `none` | the learned null token (`image_valid=False`) | what is left without imagery, the same input image dropout trained on |

The **true vs. shuffled** gap on the same buildings is the headline result, as in
`docs/ortho-conditioning-runbook.md`. It is a comparison inside one model, so it
holds even when the absolute chamfer is optimistic. The prefix arm runs on v5's
random split, where val/test buildings have training neighbours, so its absolute
chamfer is optimistic.

This is a **spot check**: `N_SHOW` buildings per split, chosen by eye. Report rates
with `src.eval.posthoc_eval ... image_mode=true|shuffled|none`. Chamfer also
moves up to ~10% from surface resampling alone (`docs/posthoc-eval-runbook.md`),
so a gap of a few percent on one building is not a finding.

Coordinate tokenizer only, as in the sibling notebook. DINOv3 is not stored in the
checkpoint, so loading downloads it from Hugging Face again. It is gated, so log in
with `huggingface-cli login` first.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import torch
from IPython.display import display
from matplotlib.collections import LineCollection

from src.dataset.mesh_datamodule import MeshDataModule
from src.dataset.mesh_dataset import (amt_detokenize, detokenize,
                                      mesh_collate_fn, specials)
from src.dataset.splits import split_name
from src.eval.mesh_metrics import mesh_metrics
from src.eval.posthoc_eval import IMAGE_MODES, generation_budget, load_model, object_seed
from src.utils.initialization import load_config
from src.visualize_mesh import cityjson_figure_from_mesh, mesh_figure, side_by_side

import plotly.io as pio

pio.renderers.default = "notebook"

## The experiment to test

Unlike the v5 notebooks, `CKPT = None` will usually fail here. The v6 runs train
remotely and nothing is under `outputs/` until you copy a checkpoint in, so set
`CKPT` to that file. Use the same `CONFIG` the run trained with: the prefix arm
for `mesh-v6-ortho-prefix-random`, or `-xattn-a/-b` for the cross-attention stages.

In [ ]:
CONFIG = ROOT / "configs" / "mesh-v6-ortho-prefix.yaml"
CKPT = None          # None = newest .ckpt under the run directory
SPLITS = ("train", "val", "test")   # which splits to generate from
N_SHOW = 4           # buildings PER SPLIT; each one is generated len(MODES) times
MODES = IMAGE_MODES  # ("true", "shuffled", "none")
TEMPERATURE = 0.0    # 0 = greedy argmax; > 0 samples with posthoc_eval's per-object seeds

cfg = load_config(CONFIG, [])

if cfg.mesh_model.tokenizer != "coord":
    raise ValueError(f"{CONFIG.name} is a '{cfg.mesh_model.tokenizer}' run -- this "
                     "notebook is the coordinate path.")
if not cfg.mesh_model.image_encoder:
    raise ValueError(f"{CONFIG.name} has no mesh_model.image_encoder -- it is an "
                     "image-free run. Use notebooks/test-mesh-normal.ipynb.")

run_dir = ROOT / cfg.logging.save_dir / cfg.logging.experiment_name / cfg.logging.run_name
if CKPT is None:
    ckpts = sorted(run_dir.rglob("*.ckpt"), key=lambda p: p.stat().st_mtime)
    if not ckpts:
        raise FileNotFoundError(f"No .ckpt under {run_dir} -- set CKPT explicitly.")
    CKPT = ckpts[-1]
CKPT = Path(CKPT)

# None of these mismatches raise later; each one gives a plausible but wrong result:
# - tokenization: decodes the sequence with the wrong inverse;
# - image_fusion / image_encoder: a different architecture would hit missing keys,
#   but `image_size` / `image_margin` would only change the crop the model was fit on;
# - data_split: the "test" buildings would not be the run's test buildings.
hp = torch.load(CKPT, map_location="cpu", weights_only=False)["hyper_parameters"]
expected = {
    "tokenization": cfg.mesh_data.tokenization,
    "image_encoder": cfg.mesh_model.image_encoder,
    "image_fusion": cfg.mesh_model.image_fusion,
    "image_size": cfg.mesh_data.image_size,
    "image_margin": cfg.mesh_data.image_margin,
    "data_split": split_name(cfg.training.split, cfg.training.split_block_m,
                             cfg.training.split_buffer_m),
}
# A checkpoint from before `data_split` existed was trained on the random split,
# which is how `load_weights_from` reads it too.
saved = {k: hp.get(k, "random" if k == "data_split" else None) for k in expected}
mismatch = {k: (saved[k], v) for k, v in expected.items() if saved[k] != v}
if mismatch:
    raise ValueError(f"{CKPT.name} vs {CONFIG.name}, (checkpoint, config): {mismatch}. "
                     "Point CONFIG at the config this checkpoint was trained with.")

print(f"config {CONFIG.name}, run {cfg.logging.run_name}")
print(f"checkpoint {CKPT}")
print(f"fusion {hp['image_fusion']}, encoder {hp['image_encoder']}, "
      f"split {expected['data_split']}, crop {hp['image_size']} px, margin {hp['image_margin']}")

## The same splits

These are the arguments `src/train_mesh.py` passes, including the ortho ones and the
split scheme. Without `ortho_dir` the "skip buildings without imagery" filter would
not run, so position *i* of a split would be a different building than in the run.

In [ ]:
m = cfg.mesh_data
datamodule = MeshDataModule(
    dataset_dir=ROOT / m.dataset_dir,
    lod_in=m.lod_in, lod_out=m.lod_out, num_bins=m.num_bins,
    margin_lo=list(m.margin_lo), margin_hi=list(m.margin_hi),
    max_faces=m.max_faces, max_files=m.max_files, tokenization=m.tokenization,
    batch_size=cfg.training.batch_size,
    train_val_test_split=tuple(cfg.training.train_val_test_split),
    num_workers=0, seed=cfg.seed,
    ortho_dir=m.ortho_dir, image_size=m.image_size, image_margin=m.image_margin,
    image_min_coverage=m.image_min_coverage,
    split=cfg.training.split, split_block_m=cfg.training.split_block_m,
    split_buffer_m=cfg.training.split_buffer_m,
)
datamodule.setup()
splits = {name: getattr(datamodule, f"{name}_dataset") for name in SPLITS}

print(f"train={len(datamodule.train_dataset)} val={len(datamodule.val_dataset)} "
      f"test={len(datamodule.test_dataset)}")
print(f"skipped without imagery: {datamodule.skipped_no_image}")

In [ ]:
# posthoc_eval's own loader, so the notebook loads the model the eval scores.
# DINOv3 is rebuilt from `image_encoder` (it is kept out of the checkpoint).
device = "cuda" if torch.cuda.is_available() else "cpu"
model = load_model(cfg, CKPT, device)
assert model.uses_image, "checkpoint is image-free despite the config"

max_new_tokens = generation_budget(cfg)   # clamped inside `generate` as well
print(f"{model.num_bins} bins, fusion {model.network.image_fusion}, "
      f"budget {max_new_tokens} tokens on {device}")

## Generate under every image mode

The condition, target and per-object seed are identical across modes; only the
image input changes. `shuffled` follows posthoc_eval's rule
(`image_source`): building *k* gets the crop of the building half the split away,
here counted over the **whole split**, not over the `N_SHOW` buildings shown.
The first few buildings of a split can sit next to each other, and pairing among
them could hand a building its neighbour's roof. The printed distance to the
source building is the check that this did not happen.

The swapped crop is the other building's own crop, in its own frame. That is what
posthoc_eval passes, so the model sees a correctly scaled roof of the wrong building.

In [ ]:
def to_metres(mesh, center, scale):
    # (verts, faces) in the unit box -> metres, undoing the LOD1-box frame.
    verts, faces = mesh
    return verts * scale + center, faces


def run_generate(batch, image, image_valid, seeds):
    with torch.no_grad():
        cond, tgt, cond_pad, _ = model._prepare(batch)
        out = model.generate(cond, cond_pad, max_new_tokens=max_new_tokens,
                             temperature=TEMPERATURE, seeds=seeds,
                             image=image, image_valid=image_valid)
    return tgt, out


cond_inverse = amt_detokenize if model.tokenization == "amt" else detokenize
results = []

for split, dataset in splits.items():
    n = min(N_SHOW, len(dataset))
    if not n:
        continue
    items = [dataset[i] for i in range(n)]
    batch = mesh_collate_fn(items, pad=specials(model.num_bins)[2])
    batch = {k: v.to(device) if torch.is_tensor(v) else v for k, v in batch.items()}
    seeds = [object_seed(cfg.seed, oid) for oid in batch["ids"]]

    # Shuffled sources: half the WHOLE split away (image_source's rule, n = len(split)).
    src_pos = [(i + len(dataset) // 2) % len(dataset) for i in range(n)]
    src_items = [dataset[j] for j in src_pos]
    shuffled = torch.stack([s["image"] for s in src_items]).to(device)

    inputs = {
        "true": (batch["image"], None),
        "shuffled": (shuffled, None),
        "none": (batch["image"], torch.zeros(n, dtype=torch.bool, device=device)),
    }
    gens = {}
    for mode in MODES:
        tgt, gens[mode] = run_generate(batch, *inputs[mode], seeds)

    for k, name in enumerate(batch["ids"]):
        center = batch["center"][k].cpu().numpy()
        scale = batch["scale"][k].cpu().numpy()
        src_center = src_items[k]["center"].numpy()
        lod1 = to_metres(cond_inverse(batch["cond"][k].cpu().numpy(), model.num_bins),
                         center, scale)
        gt = to_metres(model.decode_tokens(tgt[k]), center, scale)
        gen = {mode: to_metres(model.decode_tokens(gens[mode][k]), center, scale)
               for mode in MODES}
        results.append({
            "split": split, "name": name, "center": center, "scale": scale,
            "lod1": lod1, "gt": gt, "gen": gen,
            "crop": batch["image"][k].cpu(), "crop_shuffled": shuffled[k].cpu(),
            "src_name": src_items[k]["id"],
            "src_dist_m": float(np.linalg.norm(src_center[:2] - center[:2])),
        })
        tris = " | ".join(f"{mode} {len(gen[mode][1])}" for mode in MODES)
        print(f"{split}/{name}: lod1 {len(lod1[1])} | gt {len(gt[1])} | gen {tris} "
              f"| shuffled from {src_items[k]['id']} ({results[-1]['src_dist_m']:.0f} m away)")

## What the model sees

Crops are stored in **mesh order**: rows run along +y and columns along +x
(`src/dataset/ortho.py`). With `origin="lower"` north is up, and the axes are the
normalized frame. The crop spans `[-0.5 - margin, 0.5 + margin]` and the LOD1 box
spans `[-0.5, 0.5]`, so the meshes can be drawn on it directly.

- **Left:** the building's own crop, with the LOD1 outline (white) and the LOD2
  truth edges (cyan). If the cyan roof lines miss the roof in the photo, the
  problem is alignment or building lean, not the model. Look upstream before
  reading any metric.
- **Middle:** the same crop with the generated mesh under `true` (orange).
- **Right:** the crop the `shuffled` mode was given. Its roof should not match
  this building's outline.

In [ ]:
BETA = float(cfg.mesh_data.image_margin)
EXTENT = [-0.5 - BETA, 0.5 + BETA, -0.5 - BETA, 0.5 + BETA]


def uv_edges(mesh, center, scale):
    # Unique triangle edges projected to the normalized (u, v) plane.
    verts, faces = mesh
    if not len(faces):
        return np.zeros((0, 2, 2))
    uv = ((np.asarray(verts) - center) / scale)[:, :2]
    f = np.asarray(faces)
    e = np.sort(np.concatenate([f[:, [0, 1]], f[:, [1, 2]], f[:, [2, 0]]]), axis=1)
    e = np.unique(e, axis=0)
    return uv[e]


def show_crops(r):
    fig, axes = plt.subplots(1, 3, figsize=(13, 4.5))
    panels = [
        (r["crop"], [(r["lod1"], "white"), (r["gt"], "cyan")], "own crop: LOD1 + LOD2 truth"),
        (r["crop"], [(r["lod1"], "white"), (r["gen"]["true"], "orange")], "own crop: generated (true)"),
        (r["crop_shuffled"], [(r["lod1"], "white")],
         f"shuffled crop, from {r['src_name']} ({r['src_dist_m']:.0f} m)"),
    ]
    for ax, (img, overlays, title) in zip(axes, panels):
        ax.imshow(img.permute(1, 2, 0).numpy(), origin="lower", extent=EXTENT)
        for mesh, color in overlays:
            ax.add_collection(LineCollection(uv_edges(mesh, r["center"], r["scale"]),
                                             colors=color, linewidths=0.8))
        ax.set_xlim(EXTENT[:2]); ax.set_ylim(EXTENT[2:])
        ax.set_title(title, fontsize=9)
        ax.set_xticks([]); ax.set_yticks([])
    fig.suptitle(f"[{r['split']}] {r['name']}  "
                 f"({r['scale'][0]:.1f} x {r['scale'][1]:.1f} m LOD1 box)", fontsize=10)
    fig.tight_layout()
    plt.show()


for r in results:
    show_crops(r)

## Scores per mode

Every generation is scored against the LOD2 ground truth, through the same
tokenizer round trip as the sibling notebook. The `lod1` column is the LOD1
condition scored the same way. It is the floor of a model that copies its input
and adds nothing, so a mode that does not beat it has learned nothing about the
roof.

- `gap = shuffled - true` is the chamfer the real image is worth on that building.
  Positive means the image helped.
- `none - true` is what the model loses with no imagery at all.

`none` should land near v5 (runbook). If it is much worse, image dropout was too
low for the model to learn to work without the image.

In [ ]:
_metric = dict(taus=tuple(cfg.mesh_eval.taus), n_points=cfg.mesh_eval.n_points,
               voxel_m=cfg.mesh_eval.voxel_m)


def score(mesh, gt):
    # Empty or rejected generations score NaN instead of stopping the table.
    if not len(mesh[1]):
        return {"chamfer_m": float("nan")}
    return mesh_metrics(mesh, gt, **_metric)


long_rows, wide_rows = [], []
for r in results:
    row = {"split": r["split"], "building": r["name"],
           "lod1": score(r["lod1"], r["gt"])["chamfer_m"]}
    for mode in MODES:
        s = score(r["gen"][mode], r["gt"])
        row[mode] = s["chamfer_m"]
        long_rows.append({"split": r["split"], "building": r["name"], "mode": mode,
                          "tris": len(r["gen"][mode][1]), **s})
    row["gap"] = row["shuffled"] - row["true"]
    row["none-true"] = row["none"] - row["true"]
    row["gap_%"] = 100 * row["gap"] / row["true"] if row["true"] else float("nan")
    wide_rows.append(row)

chamfer = pd.DataFrame(wide_rows).set_index(["split", "building"])
details = pd.DataFrame(long_rows).set_index(["split", "building", "mode"])

per_split = chamfer.groupby(level="split", sort=False).mean(numeric_only=True)
print("mean chamfer (m) per split. Spot check, not a rate:")
display(per_split.round(4))
print(f"image helped on {(chamfer['gap'] > 0).sum()}/{len(chamfer)} buildings; "
      f"beat the LOD1 floor (true) on {(chamfer['true'] < chamfer['lod1']).sum()}/{len(chamfer)}")
chamfer.round(4)

In [ ]:
# Every metric mesh_metrics returns, per building and mode.
details.round(4)

## Input / truth / generated under each mode

Top row: the raw triangle meshes, one column per mode, all in the same camera.
Bottom row: the same geometry through `mesh_to_cityjson`. An empty panel means the
writer refused the mesh.

If `true` and `shuffled` look the same everywhere, the model generates from the
LOD1 prefix and ignores the image, whatever the table above says on average.

In [ ]:
def city_panel(mesh):
    # `cityjson_figure_from_mesh`, with a blank panel where it cannot run.
    if not len(mesh[1]):
        return go.Figure(), 0
    try:
        return cityjson_figure_from_mesh(*mesh)
    except Exception as exc:
        print(f"  CityJSON write-back failed: {exc}")
        return go.Figure(), 0


COLORS = {"true": "#eda100", "shuffled": "#c04a6b", "none": "#6b8e23"}
figs = {}

for r in results:
    meshes = ([(r["lod1"], "#898781", "LOD1 input"), (r["gt"], "#2a78d6", "LOD2 truth")]
              + [(r["gen"][m], COLORS[m], f"gen ({m})") for m in MODES])
    top = [mesh_figure(*mesh, color=color) for mesh, color, _ in meshes]
    bottom = [city_panel(mesh) for mesh, _, _ in meshes]
    figs[f"{r['split']}/{r['name']}"] = side_by_side(
        [top, [fig for fig, _ in bottom]],
        [f"[{r['split']}] {label}, {len(mesh[1])} tris" for mesh, _, label in meshes]
        + [f"CityJSON, {n} surfaces" if n else "CityJSON: none" for _, n in bottom],
    )

print(f"{len(figs)} figures built -- display them one at a time below.")

In [ ]:
# One at a time: each figure has ten 3-D scenes.
print(list(figs))

KEY = list(figs)[0]
figs[KEY]